# Eight-method JAX timing study

Compare all eight reference methods using the original 40 particles, reflected potential, normalized rho=0.015, 40 fixed steps and ten saved samples per cycle. Each invocation integrates **500 cycles (20,000 steps)**. One first invocation and three cached repetitions give **32 complete integrations**. All methods share one Modal CPU container (two cores, 4 GiB requested / 8 GiB limit, float64); method order alternates between rounds.

Only runtime is analyzed. No particle-distance, energy or optional step-history diagnostics are computed. Timing includes executor preparation, integration, synchronization and host result construction; it excludes input decoding, container startup, audits, files, transfers and figures. The first invocation includes compilation; it is not a separate compilation-time measurement. The worker checks one cache miss per first invocation and one cache hit per later invocation, and identical repeated states outside the timed region. See [JAX timing guidance](https://docs.jax.dev/en/latest/benchmarking.html).

This notebook persists results, individual times, parameters and source provenance in Backblaze B2. The local directory is a recovery cache. Confirmed Modal call identifiers are reused, so rerunning this notebook does not repeat finished integrations. To request a new experiment, use a new run identifier; retain the scientific settings explicitly.


In [1]:
from pathlib import Path
import json
import sys
from IPython.display import display, Image

# Locate the code repository from this section folder or its parent directories.
ROOT = next(
    (candidate.resolve()
     for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     for candidate in (parent, parent / "GC2D_codigo")
     if (candidate / "src/studies/method_timings.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "GC2D_codigo was not found. Keep it beside TFM_redaccion "
        "or start the notebook from the code repository."
    )
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

RUN_ID = "jax-20261118-rho0015-c500"
STUDY_PREFIX = "studies/method_timings"
DIRECTORY = ROOT / "outputs" / STUDY_PREFIX / RUN_ID
BUCKET = f"gc2d_data:gc2d-notebooks-data/{STUDY_PREFIX}/{RUN_ID}"
CYCLES = 500
REPETITIONS = 3


In [2]:
from diagnostics.io.storage import ArtifactStore
from studies.method_timings_io import prepare_study, recover_study, run_study

if (DIRECTORY / "study.json").exists():
    record = json.loads((DIRECTORY / "study.json").read_text())
elif ArtifactStore(BUCKET).exists("study.json"):
    record = recover_study(BUCKET, DIRECTORY)
else:
    record = prepare_study(ROOT, DIRECTORY, cycles=CYCLES, repetitions=REPETITIONS)
assert record["cycles"] == CYCLES and record["repetitions"] == REPETITIONS
assert record["study_prefix"] == STUDY_PREFIX and record["bucket"] == BUCKET
print(json.dumps(record, indent=2))


{
  "schema_version": 1,
  "study_prefix": "studies/method_timings",
  "run_id": "jax-20261118-rho0015-c500",
  "cycles": 500,
  "methods": [
    "RK2",
    "ImplicitMidpoint",
    "RK4",
    "GaussLegendre4",
    "ABBA2Midpoint",
    "ABBA2Implicit",
    "BM4Midpoint",
    "BM4Implicit"
  ],
  "repetitions": 3,
  "particle_count": 40,
  "steps_per_cycle": 40,
  "samples_per_cycle": 10,
  "rho": 0.015,
  "reference_campaign": "main-20261117-rho-div20",
  "source_sha256": "4bbd873cd05992e0a615d66ebbf4d96b38c8b3c36be2ca62932e66af7e6d4ced",
  "code_sha256": "ae8c9848787ddac8dfe1f04c251ec01378d5c6e28373cdcd8780b45f25e33794",
  "bucket": "gc2d_data:gc2d-notebooks-data/studies/method_timings/jax-20261118-rho0015-c500",
  "resources": {
    "cpu_request": 2,
    "cpu_limit": 2,
    "memory_request_mib": 4096,
    "memory_limit_mib": 8192
  },
  "timing_scope": "Execution.run: preparation, integration, synchronization and host result construction; excludes container startup, input decoding, fi

In [3]:
timings = run_study(ROOT, DIRECTORY, publish=True)
assert len(timings["measurements"]) == 8 * (REPETITIONS + 1)
assert all(m["step_count"] == CYCLES * 40 for m in timings["measurements"])
print(json.dumps({key: value for key, value in timings.items() if key != "measurements"}, indent=2))
print(json.dumps(json.loads((DIRECTORY / "publication.json").read_text()), indent=2))


{
  "payload_sha256": {
    "RK2": "c3b4dfdb6055160b5e8c47174623d9529827bc03a76c64cf7e0b650f328adea7",
    "ImplicitMidpoint": "5c4f1d7935d240093fff7f442be0940419855f911be271b8173ec06718203282",
    "RK4": "cc0875c76f6f43c06e06fe822044f9aa47c61a6c3a733c44e7198d14fff4edae",
    "GaussLegendre4": "728ac64afa242c9482d80c34c73f944478135a668e3302ae24fa100b8e6f0265",
    "ABBA2Midpoint": "ccefccf9db846d6cfc4c38df65d13b87320dbd8219434178115e578ef7b4d122",
    "ABBA2Implicit": "2988272e390de38ca938f78a11eeeb373f47e204de9c56949319f64274d674fc",
    "BM4Midpoint": "7f152477c5915c62097861c9c56d1ecaddea1a2594b85f61b57d0db8066e2ca1",
    "BM4Implicit": "987f49b68e1b6fa1808eb20227cad492945332e42f6032010f4f82e65287c4b2"
  },
  "environment": {
    "python": "3.12.10",
    "numpy": "1.26.4",
    "scipy": "1.17.1",
    "jax": "0.7.1",
    "jaxlib": "0.7.1",
    "cpu_model": "unknown",
    "platform": "Linux-4.19.0-gvisor-x86_64-with-glibc2.36",
    "x64": true,
    "visible_logical_cpus": 2,
    "devic

In [4]:
from visualization.method_timings import summarize
summary = summarize(DIRECTORY)
print(f'{"Method":20} {"First (s)":>12} {"Median (s)":>12} {"Minimum (s)":>12} {"Maximum (s)":>12}')
for row in summary["rows"]:
    print(f'{row["method"]:20} {row["first_seconds"]:12.6f} {row["median_seconds"]:12.6f} {row["minimum_seconds"]:12.6f} {row["maximum_seconds"]:12.6f}')


Method                  First (s)   Median (s)  Minimum (s)  Maximum (s)
RK2                      6.037879     2.164691     2.137034     2.284513
ImplicitMidpoint        14.050463     5.886767     5.714194     5.925600
RK4                      6.790530     3.050642     2.819637     3.155285
GaussLegendre4          16.413041     7.656109     7.651559     7.755714
ABBA2Midpoint            7.589155     4.342777     4.319757     4.632738
ABBA2Implicit           29.906867    24.468711    22.446477    24.761951
BM4Midpoint             22.847080    19.328531    19.239684    21.751638
BM4Implicit            101.012471    87.321191    81.878077    88.005103
